# Notebook 2 — Numerical Feature Engineering
Working with `telecom_customers.csv`. All numerical transformation techniques are
demonstrated on real columns from this dataset, each with a **when/why** rationale —
not just "how".

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

customers = pd.read_csv("./telecom_customers.csv", parse_dates=["signup_date"])
customers[["monthly_charges","tenure_months","total_charges"]].describe()

## 1. Basic Mathematical Transformations (Addition, Subtraction, Multiplication, Division)

These are the simplest, and most business-interpretable, engineered features. They
directly encode a business relationship between two raw numeric columns.

**When to use:** whenever two numeric columns have a *meaningful arithmetic
relationship* in the real world (e.g. price × quantity = revenue).

In [ ]:
# Ratio feature: how much of total lifetime charges does one month represent?
customers["charge_to_total_ratio"] = (
    customers["monthly_charges"] / customers["total_charges"].replace(0, np.nan)
)

# Difference feature: expected total charges (monthly * tenure) vs actual billed total
customers["expected_total_charges"] = customers["monthly_charges"] * customers["tenure_months"]
customers["billing_discrepancy"] = customers["total_charges"] - customers["expected_total_charges"]

customers[["monthly_charges","tenure_months","total_charges",
           "expected_total_charges","billing_discrepancy","charge_to_total_ratio"]].head()

**Business meaning:** `billing_discrepancy` flags customers whose actual billing history
deviates from what a simple monthly-rate model would predict — useful for both fraud/
billing-error detection and as a churn signal (surprise charges drive churn).

## 2. Percentage & Rate Features

**When to use:** when raw magnitudes differ across customers but the *relative* change
is what matters (classic in finance, telecom usage, and marketing).

In [ ]:
# Percentage of tenure spent as a "new" customer (first 6 months, higher-risk window)
customers["pct_tenure_in_risk_window"] = np.where(
    customers["tenure_months"] > 0,
    np.minimum(customers["tenure_months"], 6) / customers["tenure_months"],
    1.0
)
customers[["tenure_months","pct_tenure_in_risk_window"]].sample(5, random_state=1)

## 3. Log Transformation

**When to use:** on right-skewed numeric columns (a long tail of large values) — very
common for monetary columns. Log transformation compresses large values, stabilizes
variance, and helps linear/distance-based models a lot (tree models are less sensitive
but it still rarely hurts).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10,3.5))
axes[0].hist(customers["total_charges"].dropna(), bins=40, color="#4C72B0")
axes[0].set_title("total_charges (raw)")
customers["log_total_charges"] = np.log1p(customers["total_charges"])
axes[1].hist(customers["log_total_charges"].dropna(), bins=40, color="#DD8452")
axes[1].set_title("log1p(total_charges)")
plt.tight_layout()
plt.show()
print("Skew before:", customers['total_charges'].skew().round(3),
      "| Skew after:", customers['log_total_charges'].skew().round(3))

We use `log1p` (log(1+x)) instead of raw `log` because some engineered ratio/difference
columns can be zero, and `log(0)` is undefined.

## 4. Square Root & Power Transformations

**When to use:** milder skew correction than log, or when you specifically want to
compress the effect of large counts (e.g. number of support tickets) without fully
flattening them like log does.

In [ ]:
customers["sqrt_monthly_charges"] = np.sqrt(customers["monthly_charges"])
customers["monthly_charges_squared"] = customers["monthly_charges"] ** 2  # power transform,
# useful when the RELATIONSHIP with target is quadratic, not linear
customers[["monthly_charges","sqrt_monthly_charges","monthly_charges_squared"]].head()

## 5. Absolute Difference vs Relative Difference

- **Absolute difference** = `A - B` → meaningful when scale is consistent and the raw
  gap matters (e.g. dollars).
- **Relative difference** = `(A - B) / B` → meaningful when comparing across
  populations with very different baselines (e.g. comparing a $20/month customer to a
  $150/month customer).

In [ ]:
avg_charge_by_contract = customers.groupby("contract")["monthly_charges"].transform("mean")
customers["abs_diff_from_contract_avg"] = customers["monthly_charges"] - avg_charge_by_contract
customers["rel_diff_from_contract_avg"] = (
    customers["monthly_charges"] - avg_charge_by_contract
) / avg_charge_by_contract

customers[["contract","monthly_charges","abs_diff_from_contract_avg",
           "rel_diff_from_contract_avg"]].head()

**Business meaning:** `rel_diff_from_contract_avg` tells us whether a customer is paying
unusually *more or less* than peers on the same contract type — a much fairer
comparison than raw monthly charges, since contract types have very different price
bands.

## 6. Normalized Features

**When to use:** when a downstream model is scale-sensitive (KNN, SVM, linear models
with regularization, neural nets). Tree-based models (Random Forest, XGBoost) do **not**
need this — worth noting so you don't over-engineer.

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

num_cols = ["monthly_charges", "tenure_months"]
scaler = StandardScaler()
customers[[c + "_zscore" for c in num_cols]] = scaler.fit_transform(customers[num_cols])
customers[num_cols + [c + "_zscore" for c in num_cols]].head()

> **Leakage note:** the scaler above is fit on the *entire* dataset for demonstration
> only. In a real pipeline, `scaler.fit()` must be called **only on the training
> split**, then `.transform()` applied to validation/test — this is enforced properly
> in Notebook 13 (Pipeline).

## Summary — Feature Justification (Step 4 format)

| Feature | Source Columns | Logic | Reason | Leakage Risk | Decision |
|---|---|---|---|---|---|
| `billing_discrepancy` | monthly_charges, tenure_months, total_charges | actual − expected total | flags billing anomalies / surprise-bill churn driver | None (all pre-prediction data) | **Retain** |
| `rel_diff_from_contract_avg` | monthly_charges, contract | (x − group mean)/group mean | fair price comparison within contract type | None | **Retain** |
| `log_total_charges` | total_charges | log1p | fixes right-skew for linear models | None | **Retain** (for linear models) |
| `monthly_charges_squared` | monthly_charges | x² | test for non-linear effect | None | **Needs further analysis** — check correlation with target before keeping |